In [ ]:
import numpy as np
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

from students.miller.lesson3 import Exercise

Миллер Игорь Владиславович, ПМ-31
Lesson 3

Загрузка данных (8x8 изображения цифр)...
Размер данных: (1797, 64)
Количество классов: {len(np.unique(y))}

Размер выборок:
  Train: 1077
  Val: 360
  Test: 360

Создание модели...

Начало обучения...
Epoch 10/100 - Train Loss: 1.7823, Val Loss: 1.7432, Val Acc: 0.6833
Epoch 20/100 - Train Loss: 0.7697, Val Loss: 0.7882, Val Acc: 0.8417
Epoch 30/100 - Train Loss: 0.3641, Val Loss: 0.4243, Val Acc: 0.9167
Epoch 40/100 - Train Loss: 0.2197, Val Loss: 0.2885, Val Acc: 0.9389
Epoch 50/100 - Train Loss: 0.1547, Val Loss: 0.2259, Val Acc: 0.9417
Epoch 60/100 - Train Loss: 0.1160, Val Loss: 0.1921, Val Acc: 0.9472
Epoch 70/100 - Train Loss: 0.0924, Val Loss: 0.1690, Val Acc: 0.9472
Epoch 80/100 - Train Loss: 0.0748, Val Loss: 0.1543, Val Acc: 0.9500
Epoch 90/100 - Train Loss: 0.0622, Val Loss: 0.1442, Val Acc: 0.9556
Epoch 100/100 - Train Loss: 0.0522, Val Loss: 0.1363, Val Acc: 0.9583

Оценка модели на тестовой выборке...
Test Loss: 0.1179
Test Ac

In [ ]:
def normalize_data(X_train, X_val, X_test):

    mean = np.mean(X_train, axis=0)
    std = np.std(X_train, axis=0)

    eps = 1e-8
    std = np.where(std < eps, 1.0, std)

    X_train_norm = (X_train - mean) / std
    X_val_norm = (X_val - mean) / std
    X_test_norm = (X_test - mean) / std

    return X_train_norm, X_val_norm, X_test_norm, mean, std

In [16]:
class Training:
    def __init__(self, model, loss, learning_rate: float = 0.01):
        self.model = model
        self.loss = loss
        self.learning_rate = learning_rate

    def train_step(self, X_batch: np.ndarray, y_batch: np.ndarray) -> float:
        output = self.model.forward(X_batch)

        loss_value = self.loss.forward(output, y_batch)

        grad = self.loss.backward()
        self.model.backward(grad)

        for param, grad_param in zip(self.model.parameters, self.model.grad):
            param -= self.learning_rate * grad_param

        return loss_value

    def evaluate(self, X: np.ndarray, y: np.ndarray) -> tuple:
        output = self.model.forward(X)
        loss_value = self.loss.forward(output, y)

        predictions = np.argmax(output, axis=1)
        accuracy = np.mean(predictions == y)

        return loss_value, accuracy

    def train(
        self,
        X_train: np.ndarray,
        y_train: np.ndarray,
        X_val: np.ndarray,
        y_val: np.ndarray,
        batch_size: int = 32,
        epochs: int = 100,
        verbose: bool = True,
    ):
        n_samples = X_train.shape[0]

        for epoch in range(epochs):
            indices = np.random.permutation(n_samples)
            X_shuffled = X_train[indices]
            y_shuffled = y_train[indices]

            epoch_loss = 0
            n_batches = 0

            for i in range(0, n_samples, batch_size):
                X_batch = X_shuffled[i : i + batch_size]
                y_batch = y_shuffled[i : i + batch_size]

                batch_loss = self.train_step(X_batch, y_batch)
                epoch_loss += batch_loss
                n_batches += 1

            avg_train_loss = epoch_loss / n_batches

            val_loss, val_acc = self.evaluate(X_val, y_val)

            if verbose and (epoch + 1) % 10 == 0:
                print(
                    "Epoch {}/{} - ".format(epoch + 1, epochs)
                    + "Train Loss: {:.4f}, ".format(avg_train_loss)
                    + "Val Loss: {:.4f}, ".format(val_loss)
                    + "Val Acc: {:.4f}".format(val_acc)
                )


def create_digits_model(input_size: int = 64, hidden_sizes: list = [128, 64], output_size: int = 10):
    layers = []

    layers.append(Exercise.create_linear_layer(input_size, hidden_sizes[0]))
    layers.append(Exercise.create_relu_layer())

    for i in range(len(hidden_sizes) - 1):
        layers.append(Exercise.create_linear_layer(hidden_sizes[i], hidden_sizes[i + 1]))
        layers.append(Exercise.create_relu_layer())

    layers.append(Exercise.create_linear_layer(hidden_sizes[-1], output_size))
    layers.append(Exercise.create_logsoftmax_layer())

    return Exercise.create_model(*layers)


def main():
    print(Exercise.get_student())
    print(Exercise.get_topic())
    print("\n" + "=" * 50)
    print("Загрузка данных (8x8 изображения цифр)...")
    print("=" * 50)

    digits = load_digits()
    X = digits.data.astype(np.float32)
    y = digits.target

    print("Размер данных: {}".format(X.shape))
    print("Количество классов: {}".format(len(np.unique(y))))

    X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp)

    print("\nРазмер выборок:")
    print("  Train: {}".format(X_train.shape[0]))
    print("  Val: {}".format(X_val.shape[0]))
    print("  Test: {}".format(X_test.shape[0]))

    X_train_norm, X_val_norm, X_test_norm, mean, std = normalize_data(X_train, X_val, X_test)

    print("\n" + "=" * 50)
    print("Создание модели...")
    print("=" * 50)

    model = create_digits_model(input_size=64, hidden_sizes=[128, 64], output_size=10)

    loss = Exercise.create_nll_loss()

    training = Training(model, loss, learning_rate=0.01)

    print("\n" + "=" * 50)
    print("Начало обучения...")
    print("=" * 50)

    training.train(X_train_norm, y_train, X_val_norm, y_val, batch_size=32, epochs=100, verbose=True)

    print("\n" + "=" * 50)
    print("Оценка модели на тестовой выборке...")
    print("=" * 50)

    test_loss, test_acc = training.evaluate(X_test_norm, y_test)
    print("Test Loss: {:.4f}".format(test_loss))
    print("Test Accuracy: {:.4f}".format(test_acc))

    print("\n" + "=" * 50)
    print("Статистика по каждому классу:")
    print("=" * 50)

    output = model.forward(X_test_norm)
    predictions = np.argmax(output, axis=1)

    for digit in range(10):
        mask = y_test == digit
        if np.sum(mask) > 0:
            acc = np.mean(predictions[mask] == digit)
            print("Цифра {}: {:.4f} ({} примеров)".format(digit, acc, np.sum(mask)))

    return model, training, test_acc


if __name__ == "__main__":
    model, training, accuracy = main()

Миллер Игорь Владиславович, ПМ-31
Lesson 3

Загрузка данных (8x8 изображения цифр)...
Размер данных: (1797, 64)
Количество классов: 10

Размер выборок:
  Train: 1077
  Val: 360
  Test: 360

Создание модели...

Начало обучения...
Epoch 10/100 - Train Loss: 1.7690, Val Loss: 1.7218, Val Acc: 0.7861
Epoch 20/100 - Train Loss: 0.6877, Val Loss: 0.7014, Val Acc: 0.8611
Epoch 30/100 - Train Loss: 0.3163, Val Loss: 0.3685, Val Acc: 0.9361
Epoch 40/100 - Train Loss: 0.1944, Val Loss: 0.2515, Val Acc: 0.9556
Epoch 50/100 - Train Loss: 0.1382, Val Loss: 0.1972, Val Acc: 0.9556
Epoch 60/100 - Train Loss: 0.1038, Val Loss: 0.1664, Val Acc: 0.9639
Epoch 70/100 - Train Loss: 0.0832, Val Loss: 0.1468, Val Acc: 0.9667
Epoch 80/100 - Train Loss: 0.0680, Val Loss: 0.1340, Val Acc: 0.9667
Epoch 90/100 - Train Loss: 0.0562, Val Loss: 0.1248, Val Acc: 0.9667
Epoch 100/100 - Train Loss: 0.0478, Val Loss: 0.1176, Val Acc: 0.9667

Оценка модели на тестовой выборке...
Test Loss: 0.1225
Test Accuracy: 0.9639

С